In [49]:
import httpx
from bs4 import BeautifulSoup
import sqlite3
import re
from urllib.parse import urljoin
import time
import random

In [50]:
# Configurações
BASE_URL = "https://sites.pitt.edu/~dash/"
START_URL = urljoin(BASE_URL, "folktexts.html")
DB_NAME = "contos.sqlite"

# Adiciona um cabeçalho User-Agent para simular um navegador
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}


In [ ]:
def setup_db():
    """Cria a tabela no banco de dados SQLite."""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS  university-pittsburgh-folklore-mythology (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            titulo TEXT NOT NULL,
            origem TEXT,
            url TEXT NOT NULL,
            texto_completo TEXT,
            UNIQUE(url, titulo)
        )
    """)
    conn.commit()
    conn.close()


setup_db()

In [53]:
def fetch_page(client, url):
    """Faz a requisição HTTP usando um cliente httpx e retorna o objeto BeautifulSoup."""
    try:
        response = client.get(url, timeout=20) # Aumentando um pouco o timeout
        response.raise_for_status()
        return BeautifulSoup(response.content, 'html.parser')
    except httpx.RequestError as e:
        print(f"Erro ao acessar {url}: {e}")
        return None

In [54]:
# setup_db()
conn = None  # Inicializa a conexão como None
# try:
conn = sqlite3.connect(DB_NAME)
# Cria um cliente httpx que ignora a verificação SSL e usa os headers definidos
with httpx.Client(verify=False, headers=HEADERS) as client:
    soup = fetch_page(client, START_URL)

In [ ]:
target_urls = set()

all_links = soup.find_all('a', href=True)
for link in all_links:
    href = link['href']
    
    # Filtra links internos que terminam em .html e não são a página principal
    if href.endswith('.html') and not href.startswith('#') and not href.startswith('http') and href != 'folktexts.html':
        full_url = urljoin(BASE_URL, href)
        target_urls.add(full_url)

{'https://sites.pitt.edu/~dash/type1423.html', 'https://sites.pitt.edu/~dash/type0178a.html', 'https://sites.pitt.edu/~dash/type2033.html', 'https://sites.pitt.edu/~dash/type0505.html', 'https://sites.pitt.edu/~dash/man.html', 'https://sites.pitt.edu/~dash/type0706.html', 'https://sites.pitt.edu/~dash/type0068a.html', 'https://sites.pitt.edu/~dash/type4025.html', 'https://sites.pitt.edu/~dash/goldfowl.html', 'https://sites.pitt.edu/~dash/type5050.html', 'https://sites.pitt.edu/~dash/type0015.html', 'https://sites.pitt.edu/~dash/type0750a.html', 'https://sites.pitt.edu/~dash/norway010.html', 'https://sites.pitt.edu/~dash/type2030.html', 'https://sites.pitt.edu/~dash/type0155.html', 'https://sites.pitt.edu/~dash/edenhall.html', 'https://sites.pitt.edu/~dash/type0047e.html', 'https://sites.pitt.edu/~dash/beowulf.html', 'https://sites.pitt.edu/~dash/abduct.html', 'https://sites.pitt.edu/~dash/type2031c.html', 'https://sites.pitt.edu/~dash/type1319.html', 'https://sites.pitt.edu/~dash/havam

In [57]:
tales_pages = list(target_urls)[:10]

# Lista para armazenar os dados de cada conto
tales_data = []

with httpx.Client(verify=False, headers=HEADERS) as client:
    for tale_url in tales_pages:
        soup = fetch_page(client, tale_url)
        page_tales_data = []
        delay = random.uniform(1, 3)
        # print(f"    Aguardando {delay:.2f} segundos antes de raspar")
        time.sleep(delay)
        # print(f"Raspando {tale_url}")


        # 1. Encontrar TODOS os cabeçalhos <h2>
        all_h2s = soup.find_all('h2')

        # 2. Filtrar a lista para remover o "Contents"
        tale_headers = []
        for h2 in all_h2s:
            # Checa se existe um link <a> com name="contents" dentro do h2
            if not h2.find('a', attrs={'name': 'contents'}):
                tale_headers.append(h2)


        # 3. Iterar pela lista de cabeçalhos de contos filtrada
        for h2 in tale_headers:
            current_tale = {}
            
            # Extrair o título do conto
            current_tale['title'] = h2.get_text(strip=True)
            
            # 4. Coletar todos os elementos *dentro* deste conto
            # (ou seja, até o próximo <h2>)
            content_nodes = []
            
            # Usamos .find_next_siblings() que funciona
            # mesmo que o h2 esteja aninhado
            for sibling in h2.find_next_siblings():
                # 5. Parar quando encontrar o próximo cabeçalho h2
                if sibling.name == 'h2':
                    break
                
                # Ignora tags <hr> e <p> vazias entre os contos
                if sibling.name == 'hr':
                    continue
                if sibling.name == 'p' and not sibling.get_text(strip=True):
                    continue
                    
                content_nodes.append(sibling)
                
            # 6. Extrair os dados de dentro dos elementos do conto
            story_parts = []
            current_tale['origin'] = 'N/A' # Valor padrão

            for node in content_nodes:
                # Pula se o nó for apenas um caractere de nova linha
                if not node.name:
                    continue
                    
                if node.name == 'h3':
                    # Extrai a origem (ex: "India")
                    current_tale['origin'] = node.get_text(strip=True)
                elif node.name == 'p' or node.name == 'blockquote':
                    # Adiciona partes da história
                    story_parts.append(node.get_text(strip=True))
                

            # Junta as partes da história
            current_tale['story'] = "\n\n".join(story_parts)
            current_tale['url'] = tale_url
            
            # Adiciona o dicionário do conto à lista principal
            tales_data.append(current_tale)
            page_tales_data.append(current_tale)

In [ ]:
def insert_tale(conn, titulo, origem, url, texto_completo):
    """Insere um conto no banco de dados ou atualiza se já existir."""
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO contos (titulo, origem, url, texto_completo)
        VALUES (?, ?, ?, ?)
        ON CONFLICT(url, titulo) DO UPDATE SET
        origem=excluded.origem,
        texto_completo=excluded.texto_completo
    """, (titulo, origem, url, texto_completo))
    conn.commit()


In [ ]:
def scrape_and_save_tale(client, conn, title, origin, url, texto_completo):
    """Função auxiliar para raspar e salvar os detalhes de um único conto."""
    # Verifica se o conto já foi raspado (para evitar requisições desnecessárias)
    cursor = conn.cursor()
    cursor.execute("SELECT texto_completo FROM contos WHERE url = ? AND titulo = ?", (url, title))
    result = cursor.fetchone()
    
    # Se o conto já existe e tem conteúdo, pula.
    if result and result[0]:
        print(f"  Conto já existente e completo, pulando: {title}")
        return

    try:
        insert_tale(conn, title, origin, url, texto_completo)
        return
    except Exception as e:
        print(f"  Erro ao inserir o conto: {e}")
        return

In [61]:
for tale in tales_data:
    scrape_and_save_tale(client, conn, tale['title'], tale['origin'], tale['url'], tale['story'])

  Conto já existente e completo, pulando: The Brahman's Wife and the Mongoose
  Conto já existente e completo, pulando: The Hares and the Frogs
  Conto já existente e completo, pulando: The End of the World
  Conto já existente e completo, pulando: The Girl without Hands
  Conto já existente e completo, pulando: The Girl without Hands
  Conto já existente e completo, pulando: The Girl without Hands
  Conto já existente e completo, pulando: The Girl without Hands


In [62]:
import pandas as pd

DB_NAME = "contos.sqlite"

# Conecta ao banco de dados
conn = sqlite3.connect(DB_NAME)

# Carrega a tabela 'contos' em um DataFrame do pandas
# A consulta SQL "SELECT * FROM contos" seleciona todas as colunas e linhas
df = pd.read_sql_query("SELECT * FROM contos", conn)

# Fecha a conexão
conn.close()

# Exibe as 5 primeiras linhas do DataFrame
print("Amostra dos 5 primeiros contos:")
display(df.head())

# Exibe informações sobre o DataFrame (tipos de coluna, contagem de nulos, etc.)
print("\nInformações sobre a tabela:")
df.info()

# Exibe algumas estatísticas, como a contagem de contos por origem
print("\nContagem de contos por origem:")
display(df['origem'].value_counts())

Amostra dos 5 primeiros contos:


,id,titulo,origem,url,texto_completo
0,1,The Story of Lydia and Pyrrhus,Giovanni Boccaccio,https://sites.pitt.edu/~dash/type1423.html,"Nicostratus, a wealthy patrician, married Lydi..."
1,2,The Merchant's Tale,Geoffrey Chaucer,https://sites.pitt.edu/~dash/type1423.html,In the town of Pavia in Lombardy there lived a...
2,3,Story of the Credulous Husband,1001 Nights(translated by John Payne),https://sites.pitt.edu/~dash/type1423.html,"One day, as the woman was private with her lov..."
3,4,The Story of the Simpleton Husband,1001 Nights(translated by Richard Burton),https://sites.pitt.edu/~dash/type1423.html,"One day of the days, as the woman was closeted..."
4,5,The Twenty-Ninth Vizier's Story,Turkey,https://sites.pitt.edu/~dash/type1423.html,"There was in the palace of the world a grocer,..."



Informações sobre a tabela:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id              80 non-null     int64 
 1   titulo          80 non-null     object
 2   origem          80 non-null     object
 3   url             80 non-null     object
 4   texto_completo  80 non-null     object
dtypes: int64(1), object(4)
memory usage: 3.3+ KB

Contagem de contos por origem:


origem
Scotland                                                           8
Ireland                                                            6
Denmark                                                            6
Sweden                                                             5
Germany                                                            4
N/A                                                                4
Aesop                                                              4
India                                                              3
Russia                                                             2
Iceland                                                            2
Eskimo                                                             2
England                                                            2
Wales                                                              2
England,Gesta Romanorum                                            1
The Seven Wise Masters     